# 08 — Walk-Forward-Tuning: Symmetric Exit + Dynamic Vol Threshold/Sizing

Kopiert von `05_tuning.ipynb` (siehe README.md-Konvention). Treibt `src.backtest.tuning`
(`tune_all_baskets`/`tune_basket_params`) end-to-end: läuft eine Walk-Forward-Optuna-Suche
pro Basket über den Calibration-Zeitraum, speichert die gefundenen `best_params` unter
`data/processed/tuning/`, führt anschliessend EINEN finalen Backtest auf dem
Backtest-Zeitraum mit den getunten Parametern aus.

Unterschied zu `05_tuning.ipynb`: `BASE_PARAMS` hat `symmetric_exit=True`,
`dynamic_threshold_enabled=True`, `dynamic_sizing_enabled=True` (Opposite-Band-Exit +
Dynamic Volatility-Adjusted Thresholds/Position Sizing, siehe
docs/project/"Suggested methods...pdf" A.1/A.2/E) fest an. Gesucht wird über `entry_z` und
`entry_z_delta` (Δb der Regime-Anpassung) -- NICHT
über `exit_z` (unter `symmetric_exit=True` wirkungslos, siehe strategy.py::check_exit) und
NICHT über `vol_regime_lookback_candles`/`vol_regime_percentile_window_candles` (m/N,
theoriegeleitete Fensterparameter -- fix aus dem Sensitivitäts-Vergleich in
`01_pairwise_bollinger.ipynb` übernehmen, siehe Modul-Docstring von `src/backtest/tuning.py`
für die Data-Snooping-Begründung). `stop_z` bleibt ebenfalls fix -- erst NACH dieser Suche
anhand der gefundenen `entry_z`/`entry_z_delta` verschärfen, dann diese Suche bei Bedarf mit
dem neuen `stop_z` wiederholen.

In [69]:
import sys
from pathlib import Path
sys.path.insert(0, str(Path("../..").resolve()))

import time
import pandas as pd
from IPython.display import display

from src.config import TUNING_DIR
from src.data import load_ohlcv_panel, load_symbol_map
from src.backtest.cliques import CliqueTimeline
from src.backtest.config import BacktestConfig, BasketParams, split_calibration_backtest
from src.backtest.engine import BacktestEngine
from src.backtest import metrics as bt_metrics
from src.backtest import plots as bt_plots
from src.backtest import runs as bt_runs
from src.backtest import tuning as bt_tuning

print("Imports OK")

Imports OK


## 1. Run-Konfiguration

Der einzige Ort, den ein von diesem Template abgeleitetes Tuning-Notebook üblicherweise ändert. `BASE_PARAMS` legt alle NICHT gesuchten Parameter fest (Filter, Kosten, Positionslimits, ...) — sie bleiben für alle Trials und Baskets fix.

In [70]:
# STOP_Z: zwei Durchgaenge, 4.0 vs. 5.0 (PDF Abschnitt D) -- hier von Hand umschalten
# und Notebook zweimal laufen lassen (~3h je Durchgang, ~6h budgetiert). RUN_LABEL
# haengt am Wert UND am "full"-Suffix (siehe unten), damit dieser Durchgang NICHT mit
# dem vorherigen schnellen 16-Trial-stopz4-Test vermischt wird (andere FOLD_DAYS/
# pair_prefilter_top_n -- gleiche Study waere trotz Storage-Isolation nicht vergleichbar,
# da die Fold-Grenzen und Kandidatenpaare unterschiedlich sind).
STOP_Z = 5.0  # zweiter Durchgang von zweien -- stopz4-"full" ist bereits fertig
print(f">>> STOP_Z = {STOP_Z}  <<<")  # Sichtbarkeits-Check: sofort pruefen, ob diese Zelle
# tatsaechlich mit dem gewuenschten Wert ausgefuehrt wurde, BEVOR der Rest des Notebooks
# (jetzt ein ~3h-Lauf) startet -- siehe Notebook-Diskussion: ein frueherer Versuch lief
# versehentlich nochmal mit dem alten Wert, weil die Zellenaenderung vor dem
# Kernel-Neustart nicht ausgefuehrt/gespeichert wurde.

# entry_z-Obergrenze wird AUS stop_z HERGELEITET statt als eigene freie Zahl gesetzt --
# sonst wuerden zwei Dinge gleichzeitig variieren (stop_z UND entry_z-Spielraum), was den
# stop_z-Vergleich verwaesserte. Formel: stop_z - max(entry_z_delta) - Sicherheitspuffer,
# damit entry_z + entry_z_delta selbst im High-Vol-Regime nie an stop_z herankommt.
ENTRY_Z_DELTA_MAX = 1.0          # muss mit SEARCH_SPACE["entry_z_delta"][1] uebereinstimmen
ENTRY_Z_SAFETY_MARGIN = 0.5
ENTRY_Z_UPPER = STOP_Z - ENTRY_Z_DELTA_MAX - ENTRY_Z_SAFETY_MARGIN

RUN_LABEL = f"tune_pairwise_symmetric_dynamic_vol_full_stopz{STOP_Z:g}"
DESCRIPTION = (
    "Walk-Forward-Suche ueber entry_z/entry_z_delta, "
    "mit symmetric_exit + dynamic_threshold_enabled + dynamic_sizing_enabled fest an. "
    f"stop_z={STOP_Z} fix (PDF Abschnitt D, NICHT volatilitaetsadjustiert -- siehe "
    "PDF-Referenzcode CryptoPairsRegimeClassifier: stop_threshold bleibt dort ebenfalls "
    f"unveraendert, nur entry_mult/exit_mult sind regimeabhaengig). entry_z-Suchraum auf "
    f"(1.0, {ENTRY_Z_UPPER:g}) begrenzt -- aus stop_z hergeleitet (siehe Abschnitt 1-Kommentar "
    "zu ENTRY_Z_UPPER). 'Full'-Durchgang (robuste Einstellungen, ~3h Budget) als fairer "
    "Ersatz fuer den fruehren schnellen 8-Trial-Test -- siehe Abschnitt 1-Kommentar zu "
    "FOLD_DAYS/N_TRIALS/MIN_TRADES_PER_FOLD. m/N bleiben fix. Basket 4 ausgeschlossen "
    "(nur 10 Clique-Snapshots im gesamten Backtest-Zeitraum, Daten enden 6 Wochen vor "
    "Zeitraumende -- strukturelles Datenproblem, kein Tuning-Ziel, siehe Notebook-Diskussion)."
)

DATA_START = pd.Timestamp("2024-07-23", tz="UTC")   # frühestmögliches Datum im OHLCV-Panel
DATA_END = pd.Timestamp("2026-07-22", tz="UTC")     # spätestmögliches Datum
BACKTEST_FRACTION = 0.2  # gleiches Muster wie in _template.ipynb / 01_forward_return_screening.ipynb

# Basket 4 bewusst ausgeschlossen: nur 10 Snapshots im gesamten Backtest-Zeitraum,
# Cliquen-Daten enden am 2026-06-10 (6 Wochen vor BACKTEST_RANGE.end) -- die
# Discovery-Pipeline war fuer den vollen Zeitraum konfiguriert (siehe
# cliques_raw_24h.meta.json: date_range bis 2026-07-22), findet dort aber schlicht
# keinen Cluster mehr ueber der kalibrierten Persistenzschwelle. Kein Tuning-Problem,
# gehoert als eigener Befund/Limitation in den Bericht, nicht in diese Suche.
BASKETS = ("Basket 1", "Basket 2", "Basket 3")
CLIQUE_VARIANT = "raw_24h"  # raw|residual x 6h|12h|24h, siehe data/processed/cliques/
STRATEGY_NAME = "pairwise_zscore"  # oder "cluster_johansen"

INITIAL_CAPITAL = 100_000.0

# m/N: FIX aus dem Sensitivitaets-Vergleich in 01_pairwise_bollinger.ipynb uebernehmen
# (12h/24h/36h/48h fuer m, siehe dortige Diskussion) -- hier NICHT Teil der Optuna-Suche.
CANDLES_PER_HOUR = 12
VOL_REGIME_LOOKBACK_HOURS = 24
VOL_REGIME_LOOKBACK_CANDLES = VOL_REGIME_LOOKBACK_HOURS * CANDLES_PER_HOUR
VOL_REGIME_PERCENTILE_WINDOW_DAYS = 60
VOL_REGIME_PERCENTILE_WINDOW_CANDLES = VOL_REGIME_PERCENTILE_WINDOW_DAYS * 288

# Walk-Forward-Suche (siehe src/backtest/tuning.py fuer die volle Methodik)
# "FULL"-DURCHGANG (ersetzt den fruehren schnellen 8-Trial-Test, siehe DESCRIPTION):
# FOLD_DAYS zurueck auf 60 (8 Folds statt 5) und pair_prefilter_top_n (Abschnitt 2)
# zurueck auf 15 -- robustere Werte, jetzt wo genug Zeitbudget (~3h/Durchgang) da ist.
# N_TRIALS=25 passt bei ~115s/Trial x 3 Baskets grob in ~2.4h (Puffer fuer Schwankungen
# je Basket -- Basket 2 hat mehr Snapshots als 1/3, daher tendenziell langsamer pro
# Trial). Sicher abbrechbar: Optuna speichert jeden Trial erst nach Abschluss, ein
# Kernel-Abbruch mitten in einem Trial verliert nur DIESEN einen, nichts Frueheres.
FOLD_DAYS = 60           # Laenge/Schrittweite jedes nicht-ueberlappenden Folds
N_TRIALS = 25            # NEUE Trials in diesem Aufruf (Study wird bei gleichem RUN_LABEL fortgesetzt)
# MIN_TRADES_PER_FOLD von 1.0 auf 5.0 angehoben -- Gegenmassnahme zur beobachteten
# Objective-Pathologie (siehe tuning.py-Modul-Docstring + Notebook-Diskussion): bei
# den bisherigen Laeufen gewann durchgehend der Kandidat mit den WENIGSTEN Trades,
# weil bei den aktuellen Kosten jeder Trade im Schnitt verliert -- "kaum handeln"
# ist unter mean(fold_returns)-std(fold_returns) mathematisch am wenigsten schlecht,
# aber keine nuetzliche Handelsregel. Ein hoeherer Mindest-Trade-Filter zwingt die
# Suche, echte aktive Kandidaten zu bewerten statt sich zum "nicht handeln"-Rand
# durchzumogeln. 5.0 * 8 Folds = 40 Trades Minimum ueber die ganze Fold-Sweep.
MIN_TRADES_PER_FOLD = 5.0
SEED = 42

# Suchraum: entry_z-Obergrenze aus stop_z hergeleitet (siehe ENTRY_Z_UPPER oben, statt
# hartcodierter 2.5/3.5). entry_z_delta NEU gegenueber 05_tuning.ipynb, exit_z NICHT
# gesucht (siehe Abschnitt 0 -- unter symmetric_exit=True wirkungslos).
SEARCH_SPACE = {
    "entry_z": (0.3, ENTRY_Z_UPPER),  # GEAENDERT: war (1.0, ENTRY_Z_UPPER) -- untere Grenze war in fruehren Laeufen am Rand des Suchraums
    "entry_z_delta": (0.05, ENTRY_Z_DELTA_MAX),  # GEAENDERT: war (0.0, ...) -- Δb, PDF-Beispiel: 0.3
}

>>> STOP_Z = 5.0  <<<


In [71]:
BASE_PARAMS = BasketParams(
    # entry_z/entry_z_delta werden von der Suche
    # ueberschrieben (SEARCH_SPACE, Abschnitt 1) -- die hier gesetzten Werte sind nur
    # der Ausgangspunkt fuer den finalen Backtest, falls ein Basket keinen einzigen
    # gueltigen Trial liefert (siehe Abschnitt 5). exit_z bleibt aus Kompatibilitaets-
    # gruenden gesetzt, wird aber unter symmetric_exit=True ignoriert (siehe Abschnitt 0).
    entry_z=2.0, exit_z=0.5, stop_z=STOP_Z,
    formation_window_candles=288 * 7,
    max_candidate_pairs=3,
    # pair_prefilter_top_n zurueck auf 15 (Code-Default) -- der schnelle Testdurchgang
    # hatte das auf 8 gesenkt, jetzt (voller "Full"-Durchgang, siehe Abschnitt 1) wieder
    # die robustere volle Kandidatenbreite pro Snapshot.
    pair_prefilter_top_n=15,
    min_half_life_candles=6, max_half_life_candles=288 * 2,
    max_concurrent_positions=3,
    fee_bps=10.0, slippage_bps=2.0,
    symmetric_exit=True,  # Opposite-Band-Exit (PDF A.1), siehe strategy.py
    dynamic_threshold_enabled=True,  # Vola-Regime-abhaengige entry_z-Verschiebung (PDF A.2)
    dynamic_sizing_enabled=True,  # Vola-Regime-abhaengiges Positions-Sizing (PDF E)
    vol_regime_lookback_candles=VOL_REGIME_LOOKBACK_CANDLES,  # m, fix (siehe Abschnitt 1)
    vol_regime_percentile_window_candles=VOL_REGIME_PERCENTILE_WINDOW_CANDLES,  # N, fix
)

## 2. Daten laden

In [72]:
id_to_symbol, symbol_to_id = load_symbol_map()

# Panel muss VOR DATA_START genug Historie fuer formation_window_candles haben
# (Warm-up-Guard G7 in engine.py) -- hier grosszuegig ab DATA_START selbst geladen.
panel = load_ohlcv_panel(start=DATA_START, end=DATA_END, fields=("open", "high", "low", "close"))
print(f"Panel: {panel['close'].shape[0]:,} Kerzen x {panel['close'].shape[1]} Symbole")

Panel: 209,953 Kerzen x 47 Symbole


In [73]:
timeline = CliqueTimeline.from_cache(
    variant=CLIQUE_VARIANT, id_to_symbol=id_to_symbol, baskets=list(BASKETS),
    start=DATA_START, end=DATA_END,
)
print(timeline.describe())

Discovery-Intervall (Median über Baskets): 1 days 00:00:00
  Basket 1: 531 Snapshots, 2024-07-29 00:00:00+00:00 → 2026-07-22 00:00:00+00:00, Intervall 1 days 00:00:00
  Basket 2: 605 Snapshots, 2024-07-29 00:00:00+00:00 → 2026-07-22 00:00:00+00:00, Intervall 1 days 00:00:00
  Basket 3: 459 Snapshots, 2024-08-05 00:00:00+00:00 → 2026-06-24 00:00:00+00:00, Intervall 1 days 00:00:00


## 3. Calibration/Backtest-Split

Die Walk-Forward-Suche läuft ausschliesslich innerhalb `CALIBRATION_RANGE` (strukturell erzwungen von `generate_walkforward_folds`, siehe `src/backtest/tuning.py`) — `BACKTEST_RANGE` bleibt für den finalen Out-of-Sample-Lauf in Abschnitt 5 unberührt, exakt derselbe Split wie in jedem anderen `04_backtest`-Notebook.

In [74]:
CALIBRATION_RANGE, BACKTEST_RANGE = split_calibration_backtest(DATA_START, DATA_END, fraction=BACKTEST_FRACTION)
assert CALIBRATION_RANGE.end < BACKTEST_RANGE.start  # keine Ueberlappung, hart geprueft
print(f"Calibration (Tuning):  {CALIBRATION_RANGE.start.date()} -> {CALIBRATION_RANGE.end.date()}")
print(f"Backtest (Out-of-Sample): {BACKTEST_RANGE.start.date()} -> {BACKTEST_RANGE.end.date()}")

if RUN_LABEL == "TEMPLATE_DO_NOT_RUN":
    raise SystemExit(
        "Dies ist die Kopiervorlage -- bitte zu einem neuen Notebook kopieren "
        "(cp 05_tuning.ipynb NN_tune_kurzname.ipynb) und RUN_LABEL in Abschnitt 1 aendern, "
        "bevor die Suche ausgefuehrt wird."
    )

Calibration (Tuning):  2024-07-23 -> 2026-02-26
Backtest (Out-of-Sample): 2026-02-27 -> 2026-07-22


## 4. Walk-Forward-Suche (pro Basket)

`tune_all_baskets` läuft `tune_basket_params` für jeden Basket in Folge, mit versetztem Seed (siehe Docstring in `tuning.py` für die Begründung) und isoliert (`baskets=(basket,)` pro Fold-Engine-Lauf).

In [75]:
 #t0 = time.time()
 #tuning_results = bt_tuning.tune_all_baskets(
 #    baskets=BASKETS, panel=panel, timeline=timeline, base_params=BASE_PARAMS,
 #    calibration_range=CALIBRATION_RANGE, strategy_name=STRATEGY_NAME,
 #    initial_capital=INITIAL_CAPITAL, fold_days=FOLD_DAYS, n_trials=N_TRIALS,
 #    min_trades_per_fold=MIN_TRADES_PER_FOLD, seed=SEED, study_name=None,
 #    # storage_path an RUN_LABEL gekoppelt -- verhindert, dass Trials aus einer
 #    # frueheren Konfiguration (anderes FOLD_DAYS/SEARCH_SPACE/BASE_PARAMS) sich
 #    # mit denen dieses Laufs vermischen (siehe Notebook-Diskussion: Basket 1
 #    # hatte genau dieses Problem, 9 Alt-Trials aus einem FOLD_DAYS=30-Versuch
 #    # steckten noch im Default-Storage "optuna_walkforward.db").
 #    storage_path=TUNING_DIR / f"optuna_{RUN_LABEL}.db",
 #    search_space=SEARCH_SPACE, progress=True,
 #)
 #tuning_runtime_seconds = time.time() - t0
 #print(f"Tuning-Laufzeit: {tuning_runtime_seconds:.1f}s")

In [76]:
#BASKET_OVERRIDES = bt_tuning.build_basket_overrides(tuning_results)
#display(pd.DataFrame(BASKET_OVERRIDES).T)

## 4b. Interaktives Quick-Tuning (einzelner Basket, alle Hyperparameter)

Zusaetzliches Widget-Panel neben der automatischen Alle-Baskets-Suche oben (Abschnitt 4) --
aendert NICHTS an deren Logik/Ergebnis (`tuning_results`/`BASKET_OVERRIDES` bleiben unberuehrt).
Analog zum Panel-Pattern in aelteren Quick-Tuning-Notebooks, aber fuer EINEN Basket auf einmal (oder
automatisch fuer alle, siehe unten) und mit **allen** `BasketParams`-Feldern, die fuer `pairwise_zscore`
+ `symmetric_exit`/`dynamic_threshold`/`dynamic_sizing` relevant sind (siehe Abschnitt 0/1 dieses
Notebooks). Pro Parameter per Checkbox waehlbar: **fix** (Fix-Feld) oder **gesucht** (Bereichs-/
Auswahl-Feld). `min_half_life_candles`/`max_half_life_candles`/`max_concurrent_positions` starten mit
"tunen" bereits AN (Range-Suche als Default). `max_candidate_pairs`/`pair_prefilter_top_n` sind NICHT
mehr Teil dieses Panels (bleiben ueber `BasketParams`-Defaults aktiv).

**max_holding_days:** absoluter Kalendertage-Hard-Cap fuer die Haltedauer
(`src/backtest/config.py::BasketParams.max_holding_days`, `src/backtest/exit_rules.py::MaxHoldingDaysRule`).
Ersetzt die fruehere, half-life-relative `TimeoutRule` (`max_holding_half_life_multiplier`), die
vollstaendig aus `BasketParams`/`exit_rules.py` entfernt wurde -- eine PRO POSITION variierende
Haltedauer liess sich nicht sinnvoll als feste Tageszahl vorgeben, `max_holding_days` deckt den
Zeit-Exit jetzt allein ab.

**Zeitraum:** entweder "gesamter Kalibrierungszeitraum" (Haekchen an, deckungsgleich mit
`CALIBRATION_RANGE` aus Abschnitt 3) oder ein manuell eingegrenzter Start+Dauer-Ausschnitt fuer
zuegige Tests. **Fenstergroesse:** statt `fold_days` direkt wird die gewuenschte Fenster-**Anzahl**
(`n_windows`) gewaehlt -- `src.backtest.tuning.fold_days_for_n_windows` rechnet das in `fold_days` um
(gleiches Muster wie `make_period_window_widgets` in aelteren Quick-Tuning-Notebooks).

Ergebnis landet in `WIDGET_TUNING_RESULT` (Key = Basket) und wird zusaetzlich ueber
`src.backtest.tuning.save_tuning_results` unter `data/processed/tuning/<run_id>/` gespeichert --
unabhaengig von `tuning_results`/`tuning_run_dir` aus Abschnitt 4/5.

**Alle Kerzenlaengen automatisch durchlaufen:** Checkbox "alle Kerzenlaengen automatisch
durchlaufen" laesst beim Klick auf "Widget-Tuning starten" nacheinander ALLE Kerzenlaengen aus
`WIDGET_CANDLE_MINUTES_OPTIONS` (5/10/15/30/60/240 min) laufen statt nur der im
Kerzenlaenge-Dropdown gewaehlten.

**Alle Baskets automatisch durchlaufen (neu):** unabhaengige, zusaetzliche Checkbox, frei
kombinierbar mit der Kerzenlaengen-Automatik -- laesst nacheinander ALLE `BASKETS` laufen statt nur
des im Basket-Dropdown gewaehlten. Kombiniert mit der Kerzenlaengen-Automatik ergeben sich bis zu 4
Verhaltensweisen (ein Basket/eine Kerze, ein Basket/alle Kerzen, alle Baskets/eine Kerze, alle
Baskets/alle Kerzen) -- die Sequenz laeuft dann ueber das volle Basket x Kerzenlaenge-Kreuzprodukt.

Gedacht fuer unbeaufsichtigtes Laufenlassen (z.B. ueber Nacht), sodass am Ende fuer jede Kombination
ein eigener gespeicherter Lauf vorliegt. Schlaegt eine einzelne Kombination fehl (z.B. Mindestdauer zu
kurz, siehe Warnhinweis oben, oder zu wenige Trades fuer `min_trades_per_fold`), wird das im Output
protokolliert und die Sequenz macht mit der naechsten Kombination weiter, statt den gesamten Durchlauf
abzubrechen -- am Ende steht eine Zusammenfassung, wie viele Kombinationen erfolgreich waren.

**Zeitfenster-Slider zeigen Stunden/Tage, nicht Kerzen:** `formation_window_candles`,
`vol_regime_lookback_candles`, `vol_regime_percentile_window_candles`, `min_half_life_candles` und
`max_half_life_candles` sind intern Kerzen-ZAHLEN, bedeuten aber je nach Kerzenlaenge sehr
unterschiedlich viel Kalenderzeit -- die Slider selbst zeigen und nehmen deshalb direkt Stunden
(`min_half_life_candles`) bzw. Tage (die uebrigen vier) entgegen, nie eine Kerzenzahl. Die
Umrechnung in die Kerzenzahl der aktuell gewaehlten "Kerzenlaenge (min)" passiert erst beim Klick
auf "Widget-Tuning starten" (bzw. live fuer den Mindestdauer-Hinweis), sodass ein eingestellter
Wert bei JEDER Kerzenlaenge -- auch bei 240min (4h) -- exakt dieselbe Kalenderzeit bedeutet, ohne
dass die Slider selbst beim Wechsel von "Kerzenlaenge (min)" neu skaliert werden muessen.

In [77]:
import warnings
warnings.filterwarnings("ignore")

import ipywidgets as widgets
from IPython.display import clear_output
from dataclasses import replace as _dc_replace

import hashlib
import math

from src.data import resample_ohlcv_panel
from src.config import BINANCE_FEE_PRESETS_BPS

WIDGET_FEE_PRESETS = ("standard", "bnb_discount", "vip1", "zero_cost")
WIDGET_CANDLE_MINUTES_OPTIONS = (5, 10, 15, 30, 60, 240)
_WIDGET_LABEL_WIDTH = "230px"
_WIDGET_SLIDER_WIDTH = "260px"
_WIDGET_DESC_WIDTH = "260px"
# 5-Spalten-Grid (tunen | Name | Fix-Slider | Range-Slider | Beschreibung) statt einer
# wrap-HBox pro Zeile -- die HBox brach bei langen Kerzenzahlen (z.B. "5760 - 17280" bei
# vol_regime_percentile_window_candles) unvorhersehbar um und liess Name-Label und
# Range-Slider uebereinander rutschen. Grid haelt alle Zeilen exakt spaltenbuendig,
# unabhaengig von der Ziffernbreite einzelner Werte.
_WIDGET_GRID_TEMPLATE_COLUMNS = f"70px {_WIDGET_LABEL_WIDTH} {_WIDGET_SLIDER_WIDTH} {_WIDGET_SLIDER_WIDTH} {_WIDGET_DESC_WIDTH}"


def _widget_resolve_cost_overrides(fee_preset: str) -> dict:
    """Preset-Name -> BasketParams-Overrides (siehe src.config.BINANCE_FEE_PRESETS_BPS).
    "zero_cost" setzt zusaetzlich Slippage auf 0, nicht nur fee_bps -- analog zu
    resolve_cost_overrides in aelteren Quick-Tuning-Notebooks."""
    if fee_preset == "zero_cost":
        return {"fee_bps": 0.0, "slippage_bps": 0.0, "slippage_high_low_factor": 0.0}
    return {"fee_bps": BINANCE_FEE_PRESETS_BPS[fee_preset]}


def _widget_make_toggle_range_row(
    name: str, *, kind: str, fix_default, range_default, description: str,
    fix_min=None, fix_max=None, fix_step=None, range_min=None, range_max=None, range_step=None,
    options=None, tune_default: bool = False, candle_unit: str | None = None, display_name: str | None = None,
):
    """Baut EINE Grid-Zeile (tune_checkbox, fix_widget, range_widget, desc_html) fuer
    einen BasketParams-Feldnamen -- kompaktes Grid-Layout (siehe widget_params_grid
    unten) statt breiter HBox-Zeilen wie im ersten Entwurf, damit alle Parameter
    ohne Horizontal-Scrolling lesbar bleiben. tune_default=True startet die Zeile
    bereits im "tunen"-Modus (Bereichsfeld aktiv, Fix-Feld deaktiviert).

    candle_unit="hours"/"days": das Feld ist intern eine Kerzenzahl, der Slider
    selbst zeigt/nimmt aber Stunden bzw. Tage entgegen (FloatSlider statt
    IntSlider, auch wenn kind="int") -- die Umrechnung in eine Kerzenzahl der
    gewaehlten candle_minutes passiert ausschliesslich in
    _widget_collect_search_space(). Ohne candle_unit ist das Feld ein normaler
    Kerzen-/Wert-Slider (kind bestimmt IntSlider/FloatSlider/Dropdown).

    display_name: nur die ANGEZEIGTE Beschriftung im Grid -- name (der echte
    BasketParams-Feldname, z.B. "vol_regime_lookback_candles") bleibt der Dict-
    Key in widget_params und wird unveraendert an dataclasses.replace()/
    search_space durchgereicht. Bei candle_unit-Feldern ist der Feldname
    (endet auf "_candles") sonst irrefuehrend, weil der Slider selbst gar
    keine Kerzen mehr zeigt -- display_name ersetzt nur das Label, z.B.
    "vol_regime_lookback_days" fuer vol_regime_lookback_candles."""
    tune_checkbox = widgets.Checkbox(value=tune_default, description="tunen", indent=False, layout=widgets.Layout(width="70px"))
    name_html = widgets.HTML(f"<span style='font-family:monospace'>{display_name or name}</span>", layout=widgets.Layout(width=_WIDGET_LABEL_WIDTH))

    slider_kind = "float" if candle_unit else kind
    if slider_kind == "int":
        fix_widget = widgets.IntSlider(value=fix_default, min=fix_min, max=fix_max, step=fix_step or 1, layout=widgets.Layout(width=_WIDGET_SLIDER_WIDTH))
        range_widget = widgets.IntRangeSlider(value=range_default, min=range_min, max=range_max, step=range_step or 1, layout=widgets.Layout(width=_WIDGET_SLIDER_WIDTH))
    elif slider_kind == "float":
        fix_widget = widgets.FloatSlider(value=fix_default, min=fix_min, max=fix_max, step=fix_step or 0.01, layout=widgets.Layout(width=_WIDGET_SLIDER_WIDTH))
        range_widget = widgets.FloatRangeSlider(value=range_default, min=range_min, max=range_max, step=range_step or 0.01, layout=widgets.Layout(width=_WIDGET_SLIDER_WIDTH))
    elif slider_kind == "categorical":
        fix_widget = widgets.Dropdown(options=options, value=fix_default, layout=widgets.Layout(width=_WIDGET_SLIDER_WIDTH))
        range_widget = widgets.SelectMultiple(options=options, value=tuple(range_default), layout=widgets.Layout(width=_WIDGET_SLIDER_WIDTH, height="50px"))
    else:
        raise ValueError(f"Unbekannte kind={slider_kind!r}")

    fix_widget.disabled = tune_default
    range_widget.disabled = not tune_default

    def _on_toggle(change):
        range_widget.disabled = not change["new"]
        fix_widget.disabled = change["new"]

    tune_checkbox.observe(_on_toggle, names="value")

    desc_html = widgets.HTML(
        f"<span style='color:#888; font-size:0.8em;'>{description}</span>",
        layout=widgets.Layout(width=_WIDGET_DESC_WIDTH),
    )
    # Jedes Widget ist EINE Grid-Zelle (statt eine wrap-HBox pro Zeile, siehe
    # _WIDGET_GRID_TEMPLATE_COLUMNS oben) -- alle Zeilen bleiben dadurch spaltenbuendig,
    # auch wenn Slider-Readouts unterschiedlich breit sind (z.B. "10080" vs. "3.00").
    row = (tune_checkbox, name_html, fix_widget, range_widget, desc_html)
    # kind bleibt der LOGISCHE Typ (int/float/categorical) fuer _widget_collect_search_space
    # (bestimmt, ob am Ende gerundet auf int gemappt wird) -- candle_unit haengt separat dran,
    # damit der Aufrufer weiss, ob der Slider-Wert noch in Kerzen umgerechnet werden muss.
    return tune_checkbox, fix_widget, range_widget, kind, row, candle_unit


def _widget_collect_search_space(param_widgets: dict, candle_minutes: int) -> tuple[dict, dict]:
    """tune_checkbox=AUS -> fixed_overrides (dataclasses.replace auf BASE_PARAMS).
    tune_checkbox=AN -> search_space (tuple=Bereich bei int/float, list=Auswahl bei categorical).
    Felder mit candle_unit ("hours"/"days") werden HIER aus dem Slider-Wert (Stunden/Tage) in
    eine Kerzenzahl der aktuell gewaehlten candle_minutes umgerechnet und auf int gerundet --
    der Slider selbst kennt nur die Kalenderzeit, nie Kerzen."""
    cpd = bt_tuning.candles_per_day(candle_minutes)
    fixed_overrides, search_space = {}, {}
    for name, (tune_checkbox, fix_widget, range_widget, kind, _row, candle_unit) in param_widgets.items():
        factor = cpd if candle_unit == "days" else (cpd / 24.0 if candle_unit == "hours" else None)
        if tune_checkbox.value:
            if kind == "categorical":
                search_space[name] = list(range_widget.value)
            elif factor is not None:
                low, high = range_widget.value
                search_space[name] = (max(1, round(low * factor)), max(1, round(high * factor)))
            else:
                search_space[name] = tuple(range_widget.value)
        else:
            if factor is not None:
                fixed_overrides[name] = max(1, round(fix_widget.value * factor))
            else:
                fixed_overrides[name] = fix_widget.value
    return fixed_overrides, search_space


# Kerzenlaenge -- Basis-Panel (5min, aus Abschnitt 2 "panel") wird pro gewaehlter
# Laenge einmalig resampled und gecacht (analog get_panel_for_candle_minutes in
# get_panel_for_candle_minutes-Pattern), damit wiederholte Klicks mit derselben Laenge nicht
# jedes Mal neu resamplen.
_widget_panel_cache: dict[int, dict] = {5: panel}


def _widget_get_panel_for_candle_minutes(candle_minutes: int) -> dict:
    if candle_minutes not in _widget_panel_cache:
        _widget_panel_cache[candle_minutes] = resample_ohlcv_panel(panel, source_minutes=5, target_minutes=candle_minutes)
    return _widget_panel_cache[candle_minutes]


# --- Kopfzeile: Basket/Kerzenlaenge/Kosten -----------------------------------
w_basket = widgets.Dropdown(options=list(BASKETS), value=BASKETS[0], description="Basket:", style={"description_width": _WIDGET_LABEL_WIDTH}, layout=widgets.Layout(width="420px"))
w_candle_minutes = widgets.Dropdown(options=WIDGET_CANDLE_MINUTES_OPTIONS, value=5, description="Kerzenlaenge (min):", style={"description_width": _WIDGET_LABEL_WIDTH}, layout=widgets.Layout(width="420px"))
# alle Kerzenlaengen automatisch nacheinander durchlaufen (WIDGET_CANDLE_MINUTES_OPTIONS,
# also 5/10/15/30/60/240 min), fuer unbeaufsichtigtes Laufenlassen -- w_candle_minutes (oben)
# wird dabei ignoriert, ein einzelner Fehler bei einer Kerzenlaenge (z.B. Mindestdauer zu kurz,
# siehe w_min_duration_info) bricht die Sequenz NICHT ab, sondern wird protokolliert, die
# naechste Kerzenlaenge laeuft trotzdem (siehe _widget_on_run/_widget_run_single unten).
w_auto_all_candles = widgets.Checkbox(value=False, description="alle Kerzenlaengen automatisch durchlaufen (5/10/15/30/60/240 min)", indent=False, layout=widgets.Layout(width="520px"))
# NEU: unabhaengig davon zusaetzlich alle Baskets automatisch durchlaufen (w_basket oben wird
# dabei ignoriert) -- frei kombinierbar mit w_auto_all_candles, ergibt bis zu 4 Kombinationen
# (einzeln/einzeln, alle Kerzen/ein Basket, eine Kerze/alle Baskets, alle Kerzen/alle Baskets),
# siehe _widget_on_run.
w_auto_all_baskets = widgets.Checkbox(value=False, description="alle Baskets automatisch durchlaufen", indent=False, layout=widgets.Layout(width="520px"))
w_fee = widgets.Dropdown(options=WIDGET_FEE_PRESETS, value="standard", description="Fee-Preset:", style={"description_width": _WIDGET_LABEL_WIDTH}, layout=widgets.Layout(width="420px"))

# --- Zeitraum: gesamter Kalibrierungszeitraum ODER manueller Ausschnitt -----
w_total_timeframe = widgets.Checkbox(value=True, description="gesamter Kalibrierungszeitraum", indent=False, layout=widgets.Layout(width="320px"))
w_period_start = widgets.DatePicker(value=CALIBRATION_RANGE.start.date(), description="Start:", style={"description_width": _WIDGET_LABEL_WIDTH}, layout=widgets.Layout(width="420px"), disabled=True)
w_period_days = widgets.IntSlider(
    value=(CALIBRATION_RANGE.end - CALIBRATION_RANGE.start).days + 1, min=10,
    max=(CALIBRATION_RANGE.end - CALIBRATION_RANGE.start).days + 1,
    description="Dauer (Tage):", style={"description_width": _WIDGET_LABEL_WIDTH}, layout=widgets.Layout(width="420px"), disabled=True,
)


def _widget_on_total_timeframe_toggle(change):
    w_period_start.disabled = change["new"]
    w_period_days.disabled = change["new"]


w_total_timeframe.observe(_widget_on_total_timeframe_toggle, names="value")

# --- Mindestdauer-Hinweis: haengt von formation_window_candles/vol_regime_*
# (fix ODER Bereichs-OBERGRENZE, je nach 'tunen'-Checkbox -- worst case fuer
# die Anzeige) UND candle_minutes ab (siehe
# PairwiseZScoreStrategy.required_history_candles). Wird bei jeder relevanten
# Widget-Aenderung live neu berechnet, damit der ValueError aus
# fold_days_for_n_windows (Zeitraum < warmup_days) idealerweise gar nicht erst
# durch Klick auf "Tuning starten" entdeckt werden muss.
w_min_duration_info = widgets.HTML()


def _widget_update_min_duration_info(*_change):
    """Nutzt dieselbe Worst-Case-Logik wie tune_basket_params/
    fold_days_for_n_windows selbst (bt_tuning._required_history_candles_worst_case)
    -- garantiert exakte Konsistenz mit dem, was beim Klick auf "Widget-Tuning
    starten" tatsaechlich als warmup_days berechnet wird (siehe dortiger Fix:
    ein Trial darf formation_window_candles/vol_regime_* selbst sampeln, wenn
    diese Felder 'tunen'=AN sind -- die alte, rein lokale Schaetzung hier
    konnte davon abweichen)."""
    candle_minutes = w_candle_minutes.value
    _fixed_overrides, _search_space = _widget_collect_search_space(widget_params, candle_minutes)
    _preview_params = _dc_replace(BASE_PARAMS, **_fixed_overrides)
    strategy = bt_tuning.build_strategy(STRATEGY_NAME)
    required_candles = bt_tuning._required_history_candles_worst_case(
        strategy=strategy, base_params=_preview_params, search_space=_search_space,
    )
    warmup_days = math.ceil(required_candles / bt_tuning.candles_per_day(candle_minutes))
    min_total_days = warmup_days + w_n_windows.value  # mind. 1 Tag/Fenster nach Abzug des Warm-ups

    chosen_days = (
        (CALIBRATION_RANGE.end - CALIBRATION_RANGE.start).days + 1
        if w_total_timeframe.value else w_period_days.value
    )
    ok = chosen_days >= min_total_days
    color = "#2a7a2a" if ok else "#b02a2a"
    status = "OK" if ok else "ZU KURZ"
    w_min_duration_info.value = (
        f"<span style='color:{color}; font-size:0.85em;'>"
        f"Warm-up: {warmup_days} Tage (aus {required_candles:.0f} Kerzen @ {candle_minutes}min) + "
        f"mind. 1 Tag/Fenster (n_windows={w_n_windows.value}) &rarr; "
        f"Mindestdauer &ge; {min_total_days} Tage -- gewaehlt: {chosen_days} Tage [{status}]</span>"
    )



# --- Fenster-Anzahl statt fold_days direkt (fold_days_for_n_windows) -------
w_n_windows = widgets.IntSlider(value=8, min=1, max=30, step=1, description="n_windows:", style={"description_width": _WIDGET_LABEL_WIDTH}, layout=widgets.Layout(width="420px"))
w_trials = widgets.IntSlider(value=20, min=5, max=200, step=5, description="n_trials:", style={"description_width": _WIDGET_LABEL_WIDTH}, layout=widgets.Layout(width="420px"))
# Default 100 (vorher MIN_TRADES_PER_FOLD=5.0 aus Abschnitt 1) -- eigener Default nur fuer
# dieses Widget-Panel, Abschnitt 1 (der programmatische Lauf) bleibt unveraendert.
w_min_trades = widgets.FloatSlider(value=100.0, min=0.0, max=250.0, step=0.5, description="min_trades_per_fold:", style={"description_width": _WIDGET_LABEL_WIDTH}, layout=widgets.Layout(width="420px"))


def _widget_resolve_calibration_range() -> "bt_tuning.DateRange":
    """Liest w_total_timeframe/w_period_start/w_period_days aus -- entweder die
    volle CALIBRATION_RANGE (Abschnitt 3) oder ein daraus eingegrenzter
    [start, start+days-1]-Ausschnitt (nie groesser als CALIBRATION_RANGE)."""
    if w_total_timeframe.value:
        return CALIBRATION_RANGE
    start_ts = pd.Timestamp(w_period_start.value, tz="UTC")
    end_ts = min(start_ts + pd.Timedelta(days=w_period_days.value - 1), CALIBRATION_RANGE.end)
    start_ts = max(start_ts, CALIBRATION_RANGE.start)
    return bt_tuning.DateRange(start_ts, end_ts)


# Alle fuer diese Strategie-Konfiguration relevanten BasketParams-Felder --
# Reihenfolge/Defaults orientiert an BASE_PARAMS (Abschnitt 1). exit_z ist HIER
# bewusst NICHT tunbar -- unter symmetric_exit=True (in diesem Notebook fest an,
# siehe Abschnitt 0) ist der Exit-Zielwert immer die Gegenseite von
# entry_threshold_z (Opposite-Band-Exit, PDF A.1), exit_z wird von check_exit gar
# nicht gelesen. Ein eigener Suchraum dafuer waere reine Verschwendung von Trials.
# max_candidate_pairs/pair_prefilter_top_n wurden komplett aus dem Panel entfernt
# (siehe Notebook-Diskussion) -- beide bleiben ueber BASE_PARAMS/BasketParams-Defaults
# aktiv, sind aber hier nicht mehr einstellbar.
#
# max_holding_days: BasketParams.max_holding_days/exit_rules.MaxHoldingDaysRule --
# absoluter Kalendertage-Hard-Cap fuer die Haltedauer. Ersetzt die fruehere,
# half-life-relative TimeoutRule (max_holding_half_life_multiplier) vollstaendig --
# diese wurde aus BasketParams/exit_rules.py entfernt, da eine PRO POSITION
# variierende Haltedauer sich nicht sinnvoll als Widget-Slider "in Tagen"
# bedienen liess, waehrend max_holding_days genau das leistet.
#
# candle_unit="hours"/"days": diese Felder sind intern Kerzenzahlen, der Slider selbst
# zeigt/nimmt aber Stunden bzw. Tage entgegen (siehe _widget_make_toggle_range_row-
# Docstring) -- die Umrechnung in die Kerzenzahl der jeweils gewaehlten Kerzenlaenge
# passiert ausschliesslich in _widget_collect_search_space(), NIE am Slider selbst.
# Deshalb muss der Slider beim Wechsel der Kerzenlaenge (w_candle_minutes) NICHT neu
# skaliert werden -- anders als in einer frueheren Fassung dieses Panels, die Kerzen-
# ZAHLEN anzeigte und bei jedem Kerzenlaengen-Wechsel per Observer umgerechnet werden
# musste. display_name ersetzt bei diesen Feldern nur die ANGEZEIGTE Beschriftung
# (der echte Kerzen-Feldname bleibt der widget_params-Key), damit z.B.
# "vol_regime_lookback_days" statt "vol_regime_lookback_candles" im Grid steht --
# sonst suggeriert der Feldname faelschlich, der Slider zeige noch Kerzen.
widget_params = {
    "entry_z": _widget_make_toggle_range_row("entry_z", kind="float", fix_default=2.0, range_default=(0.3, 4.0), description="Entry-Schwelle (|Z-Score|).", fix_min=0.5, fix_max=6.0, fix_step=0.1, range_min=0.3, range_max=6.0, range_step=0.1),
    "entry_z_delta": _widget_make_toggle_range_row("entry_z_delta", kind="float", fix_default=0.3, range_default=(0.05, 1.0), description="Delta_b der Vola-Regime-Anpassung von entry_z (PDF A.2).", fix_min=0.0, fix_max=2.0, fix_step=0.05, range_min=0.0, range_max=2.0, range_step=0.05),
    "stop_z": _widget_make_toggle_range_row("stop_z", kind="float", fix_default=5.0, range_default=(3.0, 6.0), description="Stop-Loss-Schwelle (|Z-Score|), NICHT vola-adjustiert (PDF Abschnitt D).", fix_min=2.0, fix_max=8.0, fix_step=0.5, range_min=2.0, range_max=8.0, range_step=0.5),
    "max_holding_days": _widget_make_toggle_range_row("max_holding_days", kind="float", fix_default=4.0, range_default=(2.0, 10.0), description="Absoluter Hard-Cap der Haltedauer in Tagen.", fix_min=0.5, fix_max=30.0, fix_step=0.5, range_min=0.5, range_max=30.0, range_step=0.5),
    "sizing_multiplier_cap": _widget_make_toggle_range_row("sizing_multiplier_cap", kind="float", fix_default=4.0, range_default=(1.0, 5.0), description="Obergrenze des Vola-Regime-Sizing-Multiplikators (PDF E).", fix_min=1.0, fix_max=8.0, fix_step=0.25, range_min=1.0, range_max=8.0, range_step=0.25),
    "vol_regime_lookback_candles": _widget_make_toggle_range_row("vol_regime_lookback_candles", kind="int", fix_default=1.0, range_default=(100 / 288, 600 / 288), fix_min=0.05, fix_max=5.0, fix_step=0.05, range_min=0.05, range_max=5.0, range_step=0.05, candle_unit="days", display_name="vol_regime_lookback_days", description="m: Fenster fuer aktuelle Spread-Vola, fix = 1 Tag (in Tagen)."),
    "vol_regime_percentile_window_candles": _widget_make_toggle_range_row("vol_regime_percentile_window_candles", kind="int", fix_default=35.0, range_default=(20.0, 60.0), fix_min=10.0, fix_max=180.0, fix_step=1.0, range_min=10.0, range_max=180.0, range_step=1.0, candle_unit="days", display_name="vol_regime_percentile_window_days", description="N: Fenster fuer Vola-Regime-Perzentile, fix = 35 Tage (in Tagen)."),
    "vol_regime_low_percentile": _widget_make_toggle_range_row("vol_regime_low_percentile", kind="float", fix_default=20.0, range_default=(10.0, 30.0), description="Perzentilschwelle Low-Vol-Regime.", fix_min=1.0, fix_max=49.0, fix_step=1.0, range_min=1.0, range_max=49.0, range_step=1.0),
    "vol_regime_high_percentile": _widget_make_toggle_range_row("vol_regime_high_percentile", kind="float", fix_default=80.0, range_default=(70.0, 90.0), description="Perzentilschwelle High-Vol-Regime.", fix_min=51.0, fix_max=99.0, fix_step=1.0, range_min=51.0, range_max=99.0, range_step=1.0),
    "formation_window_candles": _widget_make_toggle_range_row("formation_window_candles", kind="int", fix_default=20.0, range_default=(10.0, 30.0), fix_min=1.0, fix_max=45.0, fix_step=1.0, range_min=1.0, range_max=45.0, range_step=1.0, candle_unit="days", display_name="formation_window_days", description="Kalibrierungsfenster fuer Hedge-Ratio/Cointegration bei Formation, fix = 20 Tage (in Tagen)."),
    "min_half_life_candles": _widget_make_toggle_range_row("min_half_life_candles", kind="int", fix_default=3.0, range_default=(1.0, 6.0), fix_min=0.25, fix_max=24.0, fix_step=0.25, range_min=1.0, range_max=6.0, range_step=0.25, candle_unit="hours", display_name="min_half_life_hours", description="Untere Half-Life-Schranke fuer gueltige Spreads (in Stunden).", tune_default=True),
    "max_half_life_candles": _widget_make_toggle_range_row("max_half_life_candles", kind="int", fix_default=4.0, range_default=(3.0, 15.0), fix_min=0.5, fix_max=30.0, fix_step=0.5, range_min=3.0, range_max=15.0, range_step=0.5, candle_unit="days", display_name="max_half_life_days", description="Obere Half-Life-Schranke fuer gueltige Spreads (in Tagen).", tune_default=True),
    "max_concurrent_positions": _widget_make_toggle_range_row("max_concurrent_positions", kind="int", fix_default=3, range_default=(1, 15), description="Max. gleichzeitig offene Positionen je Basket.", fix_min=1, fix_max=20, range_min=1, range_max=20, tune_default=True),
}

# Observer erst NACH widget_params registrieren (braucht die Checkbox-/Slider-
# Objekte fuer _widget_field_worst_case) -- deckt sowohl die Kopf-Widgets
# (Kerzenlaenge/Zeitraum/n_windows) als auch die relevanten Parameter-Zeilen ab.
for _w in (w_candle_minutes, w_total_timeframe, w_period_days, w_n_windows):
    _w.observe(_widget_update_min_duration_info, names="value")
for _field_name in ("formation_window_candles", "vol_regime_percentile_window_candles", "vol_regime_lookback_candles"):
    _tune_checkbox, _fix_widget, _range_widget, _kind, _row, _candle_unit = widget_params[_field_name]
    _tune_checkbox.observe(_widget_update_min_duration_info, names="value")
    _fix_widget.observe(_widget_update_min_duration_info, names="value")
    _range_widget.observe(_widget_update_min_duration_info, names="value")
_widget_update_min_duration_info()

widget_run_button = widgets.Button(description="Widget-Tuning starten", button_style="primary", icon="bolt")
widget_output = widgets.Output()
WIDGET_TUNING_RESULT = {}


def _widget_run_single(basket: str, candle_minutes: int) -> None:
    """EIN kompletter Tuning-Durchlauf fuer genau einen Basket und eine
    Kerzenlaenge -- eigene Funktion, damit _widget_on_run sie sowohl fuer den
    manuellen Einzel-Lauf (w_basket/w_candle_minutes) als auch fuer die
    automatischen Sequenzen ueber WIDGET_CANDLE_MINUTES_OPTIONS
    (w_auto_all_candles) und/oder BASKETS (w_auto_all_baskets) wiederverwenden
    kann, ohne die Ablauflogik zu duplizieren."""
    # candle_minutes wird HIER direkt an _widget_collect_search_space durchgereicht --
    # Stunden-/Tage-Felder (candle_unit) werden dort in die Kerzenzahl DIESER Kerzenlaenge
    # umgerechnet, unabhaengig davon ob w_candle_minutes.value je gesetzt wurde (die
    # Alle-Kerzenlaengen-Auto-Sequenz ruft _widget_run_single fuer jede Laenge direkt auf).
    fixed_overrides, search_space = _widget_collect_search_space(widget_params, candle_minutes)
    base_params = _dc_replace(
        BasketParams(
            **_widget_resolve_cost_overrides(w_fee.value),
            symmetric_exit=BASE_PARAMS.symmetric_exit,
            dynamic_threshold_enabled=BASE_PARAMS.dynamic_threshold_enabled,
            dynamic_sizing_enabled=BASE_PARAMS.dynamic_sizing_enabled,
        ),
        **fixed_overrides,
    )
    widget_panel = _widget_get_panel_for_candle_minutes(candle_minutes)
    effective_range = _widget_resolve_calibration_range()
    fold_days = bt_tuning.fold_days_for_n_windows(
        effective_range, n_windows=w_n_windows.value, base_params=base_params,
        strategy_name=STRATEGY_NAME, candle_minutes=candle_minutes,
        search_space=search_space,  # worst-case warmup_days, falls formation_window_candles/
                                     # vol_regime_* selbst gesucht werden (siehe src/backtest/tuning.py-Fix)
    )

    print(f"Basket: {basket} | Kerzenlaenge: {candle_minutes}min | fee_preset: {w_fee.value} | "
          f"fixed: {fixed_overrides} | search_space: {search_space}")
    print(f"Zeitraum: {effective_range.start.date()} -> {effective_range.end.date()} | "
          f"n_windows={w_n_windows.value} -> fold_days={fold_days}\n")

    # search_space-Hash im Study-Namen: ein bereits existierender Study unter diesem Namen
    # laedt in Optuna gespeicherte ALTE Trials nach -- stimmt deren Parameter-Set (welche
    # Felder ueberhaupt gesucht wurden) nicht mit dem AKTUELLEN search_space ueberein, wirft
    # Optuna beim naechsten trial.suggest_*() "Cannot set different distribution kind to the
    # same parameter name." (z.B. wenn min_half_life_candles frueher fix war und jetzt gesucht
    # wird). Der Hash macht jede unterschiedliche Suchraum-Kombination zu einer eigenen Study,
    # ohne den fuer Menschen lesbaren Namen selbst aufzublaehen.
    search_space_fingerprint = hashlib.sha1(
        ",".join(sorted(search_space)).encode()
    ).hexdigest()[:8]
    study_name = f"{RUN_LABEL}-widget-{basket}-{candle_minutes}min-{w_fee.value}-{search_space_fingerprint}"
    storage_path = bt_tuning.TUNING_DIR / "optuna_walkforward.db"
    bt_tuning.ensure_dashboard_running(storage_path, port=bt_tuning.DASHBOARD_PORTS.get("pairwise_zscore"))

    result = bt_tuning.tune_basket_params(
        basket=basket, panel=widget_panel, timeline=timeline, base_params=base_params,
        calibration_range=effective_range, strategy_name=STRATEGY_NAME,
        fold_days=fold_days, n_trials=w_trials.value, candle_minutes=candle_minutes,
        min_trades_per_fold=w_min_trades.value, search_space=search_space or None,
        study_name=study_name, storage_path=storage_path,
        progress=True,
    )
    print(f"\nFertig. Bestes Objective: {result.study.best_value:.4f} ueber {len(result.trials_df)} Trials, {len(result.folds)} Folds")
    print(f"Beste Parameter: {result.best_params}")
    display(result.trials_df.sort_values("objective", ascending=False).head(5))

    run_dir = bt_tuning.save_tuning_results(
        {basket: result}, label=f"{RUN_LABEL}-widget-{basket}-{candle_minutes}min-{w_fee.value}",
        candle_minutes=candle_minutes, fee_preset=w_fee.value,
    )
    print(f"Gespeichert: {run_dir}")
    WIDGET_TUNING_RESULT[basket] = result


def _widget_on_run(_):
    with widget_output:
        clear_output(wait=True)
        baskets = list(BASKETS) if w_auto_all_baskets.value else [w_basket.value]
        candle_minutes_list = list(WIDGET_CANDLE_MINUTES_OPTIONS) if w_auto_all_candles.value else [w_candle_minutes.value]

        if len(baskets) == 1 and len(candle_minutes_list) == 1:
            _widget_run_single(baskets[0], candle_minutes_list[0])
            return

        # --- Automatische Sequenz ueber Basket(s) x Kerzenlaenge(n) -- fuer
        # unbeaufsichtigtes Laufenlassen (z.B. ueber Nacht). Ein Fehlschlag bei
        # einer einzelnen Kombination (z.B. Mindestdauer zu kurz bei sehr
        # grossem formation_window_candles, siehe w_min_duration_info, oder ein
        # RuntimeError aus dem min_trades_per_fold-Filter) wird protokolliert
        # und die Sequenz macht mit der naechsten Kombination weiter, statt
        # komplett abzubrechen.
        failures: list[tuple[str, int, str]] = []
        combos = [(basket, cm) for basket in baskets for cm in candle_minutes_list]
        n_total = len(combos)
        for i, (basket, candle_minutes) in enumerate(combos, start=1):
            print(f"{'=' * 70}\n[{i}/{n_total}] Basket: {basket} | Kerzenlaenge: {candle_minutes}min\n{'=' * 70}")
            try:
                _widget_run_single(basket, candle_minutes)
            except Exception as exc:  # noqa: BLE001 -- bewusst breit, siehe Docstring oben
                failures.append((basket, candle_minutes, f"{type(exc).__name__}: {exc}"))
                print(f"FEHLER bei {basket}/{candle_minutes}min -- wird uebersprungen: {type(exc).__name__}: {exc}")
            print()

        print(f"{'=' * 70}\nAutomatischer Durchlauf fertig: {n_total - len(failures)}/{n_total} Kombinationen erfolgreich.")
        if failures:
            print("Fehlgeschlagen:")
            for basket, candle_minutes, msg in failures:
                print(f"  - {basket}/{candle_minutes}min: {msg}")


widget_run_button._click_handlers.callbacks.clear()
widget_run_button.on_click(_widget_on_run)

# Kopfzeile der Parameter-Tabelle + eine Grid-Zelle je Widget (5 Spalten x len(widget_params)
# Zeilen) -- GridBox statt VBox-von-HBoxen, siehe _widget_make_toggle_range_row-Docstring.
_widget_params_header = [
    widgets.HTML("<b>tunen</b>"), widgets.HTML("<b>Parameter</b>"),
    widgets.HTML("<b>fix</b>"), widgets.HTML("<b>Bereich</b>"), widgets.HTML("<b>Beschreibung</b>"),
]
_widget_params_grid = widgets.GridBox(
    _widget_params_header + [cell for p in widget_params.values() for cell in p[4]],
    layout=widgets.Layout(
        grid_template_columns=_WIDGET_GRID_TEMPLATE_COLUMNS,
        grid_gap="4px 12px",
        align_items="center",
    ),
)

display(widgets.VBox([
    widgets.HTML("<b>Interaktives Quick-Tuning (pairwise_zscore, symmetric_exit + dynamic_threshold + dynamic_sizing)</b>"),
    widgets.HTML("<u>Basket, Kerzenlaenge & Kosten</u>"),
    w_basket, w_candle_minutes, w_auto_all_candles, w_auto_all_baskets, w_fee,
    widgets.HTML("<u>Zeitraum & Suchbudget</u>"),
    w_total_timeframe, w_period_start, w_period_days, w_n_windows, w_trials, w_min_trades,
    w_min_duration_info,
    widgets.HTML("<u>Parameter</u> (Haekchen 'tunen' AN = Suche im Bereichsfeld, AUS = fix im linken Feld)"),
    _widget_params_grid,
    widget_run_button, widget_output,
]))


## 5. Tuning-Ergebnisse speichern

Best-Params + volle `trials_df` je Basket unter `data/processed/tuning/<run_id>/` (analog zu `src.backtest.runs.save_run`, siehe `src.backtest.tuning.save_tuning_results`) — unabhängig vom finalen Backtest-Run in Abschnitt 7, damit die Suche selbst nachvollziehbar bleibt (z.B. wie stabil `best_value` über die Trials war).

In [78]:
#tuning_run_dir = bt_tuning.save_tuning_results(tuning_results, label=RUN_LABEL)
#print(f"Tuning-Ergebnisse gespeichert: {tuning_run_dir}")
#
#for basket, result in tuning_results.items():
#    print(f"\n{basket}: bestes Objective={result.study.best_value:.4f} ueber {len(result.trials_df)} Trials, {len(result.folds)} Folds")
#    display(result.trials_df.sort_values("objective", ascending=False).head(5))

## 6. Finaler Out-of-Sample-Backtest mit getunten Parametern

Exakt derselbe Ablauf wie in jedem anderen `04_backtest`-Notebook (siehe `_template.ipynb`), nur mit `basket_overrides=BASKET_OVERRIDES` aus der Suche statt Hand-getunten Werten.

In [79]:
#config = BacktestConfig(
#    label=RUN_LABEL, description=DESCRIPTION or f"Walk-Forward-getunt ({N_TRIALS} Trials/Basket, {FOLD_DAYS}d Folds)",
#    calibration_range=CALIBRATION_RANGE, backtest_range=BACKTEST_RANGE,
#    baskets=BASKETS, clique_variant=CLIQUE_VARIANT, strategy_name=STRATEGY_NAME,
#    default_params=BASE_PARAMS, basket_overrides=BASKET_OVERRIDES,
#    initial_capital=INITIAL_CAPITAL,
#)
#
#engine = BacktestEngine(config=config, panel=panel, timeline=timeline)
#print(engine.describe_guards())

In [80]:
#t0 = time.time()
#result = engine.run(progress=True)
#runtime_seconds = time.time() - t0
#print(f"Laufzeit: {runtime_seconds:.1f}s | {len(result.trades)} Trades | Warnungen: {len(engine.warnings)}")

## 7. Metriken

In [81]:
#BENCHMARK_SYMBOL = config.benchmark_symbol
#benchmark_close = panel["close"][BENCHMARK_SYMBOL] if BENCHMARK_SYMBOL in panel["close"].columns else None
#
#result.metrics = bt_metrics.compute_all_metrics(
#    equity_curve=result.equity_curve, trades=result.trades,
#    benchmark_close=benchmark_close, config=config,
#)
#result.metrics_by_basket = bt_metrics.metrics_by_basket(result.trades, config)
#result.meta["runtime_seconds"] = runtime_seconds
#result.meta["tuning_run_dir"] = str(tuning_run_dir)
#
#display(result.summary())
#display(result.metrics_by_basket)

## 8. Equity-Kurve & Drawdown

In [82]:
#bt_plots.plot_equity_and_drawdown(result, benchmark_close=benchmark_close)

## 9. Run speichern & mit früheren Runs vergleichen

In [83]:
#run_dir = bt_runs.save_run(result)
#print(f"Gespeichert: {result.run_id}")
#print(run_dir)

In [84]:
#display(bt_runs.list_runs(refresh_index=True).tail(10))

In [85]:
# Konkrete Runs zum Vergleich eintragen, z.B. getunt vs. hand-parametrisiert:
# bt_runs.compare_runs([result.run_id, "<anderer-run-id>"])

## 10. Interpretation

_Freitext: wie stabil war `best_value` über die Trials (siehe `trials_df` in Abschnitt 5)? Wich der finale Out-of-Sample-Backtest stark von der Walk-Forward-Robustheitsschätzung ab (Overfitting-Indiz)? Welche Baskets profitierten am meisten vom Tuning gegenüber Hand-Parametern? Wie gross ist das gefundene `entry_z_delta` typischerweise (nahe 0 = Regime-Anpassung bringt kaum etwas, nahe am oberen Rand des Suchraums = evtl. Suchraum erweitern)?_

### stop_z=4.0 vs. stop_z=5.0
Nach beiden "full"-Durchgängen (`STOP_Z=4.0` und `STOP_Z=5.0`, Abschnitt 1) über `04_run_comparison.ipynb` vergleichen (`run_id`s beider `RUN_LABEL`-Varianten eintragen) — insbesondere `exit_reason_counts`/Stop-Rate und `sharpe_daily`/`total_return`. Welcher Wert führt zu weniger Stop-Exits, ohne dass die Verluste pro Stop-Trade dafür deutlich grösser werden? Die früheren schnellen 8-Trial-Läufe (`_stopz4`/`_stopz5` ohne `_full`) bewusst NICHT mehr für den Vergleich heranziehen — andere `FOLD_DAYS`/`pair_prefilter_top_n`, nicht vergleichbar.

### Nächste Schritte
- [ ] Besseren `stop_z`-Wert übernehmen (basierend auf den beiden `_full`-Läufen).
- [ ] Falls im m-Sensitivitätsvergleich (`01_pairwise_bollinger.ipynb`) ein anderes `VOL_REGIME_LOOKBACK_HOURS` als das hier eingetragene gewonnen hat: hier nachziehen und neu tunen.